In [54]:
import re
import pandas as pd

def normalize_peptide(peptide):
        # Remove modification annotations for comparison
        # Remove modification annotations and all numbers for comparison
        return re.sub(r'[-.+\d]', '', peptide)

def read_mgf_peptides(mgf_file):
    """
    Read peptides and their charge states from an MGF file.

    Supports keys: SEQ/PEPTIDE for peptide, CHARGE, and SCAN/SCANS/PROVENANCE_SCAN/TITLE for scan.

    Args:
        mgf_file (str): Path to the MGF file.
    
    Returns:
        dict: A dictionary with scan as keys, peptide sequences, charge status as values.
    """
    def parse_charge(value: str):
        m = re.search(r'(\d+)', value)
        return int(m.group(1)) if m else None

    def scan_from_title(value: str):
        m = re.search(r'scan[:=](\d+)', value, re.IGNORECASE)
        return m.group(1) if m else None

    mgf_peptides = {}
    with open(mgf_file, 'r') as f:
        scan = None
        peptide = None
        charge = None
        for raw_line in f:
            line = raw_line.strip()
            if not line:
                continue

            if line.startswith('BEGIN IONS'):
                scan = None
                peptide = None
                charge = None
                continue

            if line.startswith('END IONS'):
                if scan and peptide and charge is not None:
                    mgf_peptides[str(scan)] = (peptide, charge)
                scan = None
                peptide = None
                charge = None
                continue

            if '=' in line:
                key, val = line.split('=', 1)
                key = key.strip().upper()
                val = val.strip()

                if key in ('SCAN', 'SCANS') and not scan:
                    scan = val
                elif key == 'PROVENANCE_SCAN' and not scan:
                    m = re.search(r'(\d+)', val)
                    if m:
                        scan = m.group(1)
                elif key == 'TITLE' and not scan:
                    s = scan_from_title(val)
                    if s:
                        scan = s
                elif key in ('SEQ', 'PEPTIDE'):
                    peptide = val
                elif key == 'CHARGE':
                    charge = parse_charge(val)
    return mgf_peptides




def psm_mgf_match(mgf_peptides, psm_file):
    """
    Match peptides from MGF and PSM files based on scan, unmodified peptide sequence, and charge state.

    Returns:
        filtered psm_file in tsv: going thorugh psm tsv file rows, find same scan, check if peptide unmod and charge states are the same, if same save to a the output
    """
    psm_df = pd.read_csv(psm_file, sep='\t', keep_default_na=False, na_values=['N/A'])

    matched_rows = []
    for index, row in psm_df.iterrows():
        scan = str(row['Scan'])
        psm_peptide = str(row['Annotation'])
        psm_charge = str(row['Charge'])

        if scan in mgf_peptides:
            mgf_peptide, mgf_charge = mgf_peptides[scan]
            if (normalize_peptide(psm_peptide) == normalize_peptide(mgf_peptide)) and (int(psm_charge) == int(mgf_charge)):
                matched_rows.append(row)

    matched_df = pd.DataFrame(matched_rows)
    output_file = 'matched_psm_output.tsv'
    matched_df.to_csv(output_file, sep='\t', index=False, na_rep='N/A')
    print(f"Matched PSM entries saved to {output_file}")

    


In [55]:
# Paths can be overridden if mgf_file/psm_file are already defined elsewhere
mgf_file = 'data/Normal_1_2_3.mgf'
psm_file = 'data/precursor_search_output_psms.tsv'

# Read peptides from MGF and PSM files
try:
    mgf_peptides = read_mgf_peptides(mgf_file)
    print(f"Loaded {len(mgf_peptides)} MGF entries from {mgf_file}")
except FileNotFoundError:
    print(f"MGF file not found: {mgf_file}")


Loaded 24559 MGF entries from data/Normal_1_2_3.mgf


In [56]:
missing_keys = [str(i) for i in range(1, 26192) if str(i) not in mgf_peptides]
print(f"Missing keys: {len(missing_keys)} out of {26191}")
print(f"First 20 missing keys: {missing_keys[:20]}")

Missing keys: 1632 out of 26191
First 20 missing keys: ['118', '119', '120', '121', '159', '163', '166', '168', '170', '171', '172', '173', '174', '180', '181', '182', '183', '184', '185', '186']


In [57]:
psm_mgf_match(mgf_peptides, psm_file)

Matched PSM entries saved to matched_psm_output.tsv
